# Chapter 4 Exercises
## Exercise 4.1

Calculate and compare the number of parameters that are contained in the feed forward module and those that are contained in the multi-head attention module.

In [4]:
from gpt_2.model import (
    GPT_CONFIG_124M,
    GPTModel
)
gpt2_small = GPTModel(GPT_CONFIG_124M)
mha = gpt2_small.trf_blocks[0].att
ff = gpt2_small.trf_blocks[0].ff
n_param_mha = sum(p.numel() for p in mha.parameters())
n_param_ff = sum(p.numel() for p in ff.parameters())
print(f"Number of parameters inside a single multi-head attention module: {n_param_mha:,}")
print(f"Number of parameters inside a single feed forward module: {n_param_ff:,}")

Number of parameters inside a single multi-head attention module: 2,360,064
Number of parameters inside a single feed forward module: 4,722,432


## Exercise 4.2

We initialized a 124-million-parameter GPT model, which is known as “GPT-2 small.” Without making any code modifications besides updating the configuration file, use the GPTModel class to implement GPT-2 medium (using 1,024-dimensional embeddings, 24 transformer blocks, 16 multi-head attention heads), GPT-2 large (1,280-dimensional embeddings, 36 transformer blocks, 20 multi-head attention heads), and GPT-2 XL (1,600-dimensional embeddings, 48 transformer blocks, 25 multi-head attention heads). As a bonus, calculate the total number of parameters in each GPT model.

In [9]:
GPT_CONFIG_MID = GPT_CONFIG_124M.copy()
GPT_CONFIG_MID['emb_dim'] = 1024
GPT_CONFIG_MID['n_layers'] = 24
GPT_CONFIG_MID['n_heads'] = 16

gpt2_medium = GPTModel(GPT_CONFIG_MID)
n_params = sum(p.numel() for p in gpt2_medium.parameters())
total_size_mb = (n_params * 4) // (1024 * 1024)
print(f"GPT-2-MEDIUM number of parameters: {n_params:,}")
print(f"GPT-2-MEDIUM total size (MB): {total_size_mb:,}")
del gpt2_medium

GPT_CONFIG_LARGE = GPT_CONFIG_124M.copy()
GPT_CONFIG_LARGE['emb_dim'] = 1280
GPT_CONFIG_LARGE['n_layers'] = 36
GPT_CONFIG_LARGE['n_heads'] = 20

gpt2_large = GPTModel(GPT_CONFIG_LARGE)
n_params = sum(p.numel() for p in gpt2_large.parameters())
total_size_mb = (n_params * 4) // (1024 * 1024)
print(f"\nGPT-2-LARGE number of parameters: {n_params:,}")
print(f"GPT-2-LARGE total size (MB): {total_size_mb:,}")
del gpt2_large

GPT_CONFIG_XL = GPT_CONFIG_124M.copy()
GPT_CONFIG_XL['emb_dim'] = 1600
GPT_CONFIG_XL['n_layers'] = 48
GPT_CONFIG_XL['n_heads'] = 25

gpt2_xl = GPTModel(GPT_CONFIG_XL)
n_params = sum(p.numel() for p in gpt2_xl.parameters())
total_size_mb = (n_params * 4) // (1024 * 1024)
print(f"\nGPT-2-XL number of parameters: {n_params:,}")
print(f"GPT-2-XL total size (MB): {total_size_mb:,}")
del gpt2_xl


GPT-2-MEDIUM number of parameters: 406,212,608
GPT-2-MEDIUM total size (MB): 1,549

GPT-2-LARGE number of parameters: 838,220,800
GPT-2-LARGE total size (MB): 3,197

GPT-2-XL number of parameters: 1,637,792,000
GPT-2-XL total size (MB): 6,247


## Exercise 4.3

At the beginning of this chapter, we defined a global drop_rate setting in the GPT_ CONFIG_124M dictionary to set the dropout rate in various places throughout the GPTModel architecture. Change the code to specify a separate dropout value for the various dropout layers throughout the model architecture. (Hint: there are three distinct places where we used dropout layers: the embedding layer, shortcut layer, and multi-head attention module.)

In [23]:
import torch.nn as nn
from gpt_2.model import (
    MultiHeadAttention,
    FeedForward,
    LayerNorm,)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate_attn"],      #1
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(        #2
            cfg["drop_rate_shortcut"]           #2
        )                                       #2

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


cfg = GPT_CONFIG_124M.copy()
cfg['drop_rate_shortcut'] = 0.2
cfg['drop_rate_attn'] = 0.3

mod_trnsf = TransformerBlock(cfg)
dropout_attn = mod_trnsf.att.dropout
dropout_shortcut = mod_trnsf.drop_shortcut
print(f'dropout_attn: {dropout_attn}')
print(f'dropout_shortcut: {dropout_shortcut}')


dropout_attn: Dropout(p=0.3, inplace=False)
dropout_shortcut: Dropout(p=0.2, inplace=False)
